# Phase 5, Pipeline A — Step 2: VLM verification of LAION candidates

Runs the Qwen2.5-VL judge on the candidates collected by `exp5_collect_laion.py`.

**For the pilot (30 candidates):** ~5 minutes on A100, ~15 on L4.

**Output:** approved.csv (clean training pool) and rejected.csv (useful for debugging the collection logic).

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"
CANDIDATES_DRIVE = "/content/drive/MyDrive/binding-research/finetuning/candidates"

import os, subprocess
REPO_DIR = "/projects/challenges-in-attribute-control"  
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

assert os.path.exists(CANDIDATES_DRIVE), f'Upload candidates to {CANDIDATES_DRIVE} first'

## 2. Install deps

In [ ]:
!pip install -q uv
!uv pip install -e . --system

## 3. HF login (Qwen2.5-VL requires accepting its license)

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm GPU is correct (A100/L4 — needs ~16 GB VRAM)

In [ ]:
import torch
assert torch.cuda.is_available()
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name} ({vram:.1f} GB)')
assert vram >= 20, 'Switch to A100 or L4 (Qwen-VL-7B needs ~16 GB)'

## 5. Symlink candidates and run verification

In [ ]:
!mkdir -p data/finetuning && ln -sfn $CANDIDATES_DRIVE data/finetuning/candidates
!ls data/finetuning/candidates/candidates_manifest.csv

!python experiments/exp5_verify_candidates.py \
    --candidates-root data/finetuning/candidates \
    --out-root /content/verified \
    --config configs/judge_default.yaml

## 6. Inspect results

In [ ]:
import pandas as pd
appr = pd.read_csv('/content/verified/approved.csv')
rej  = pd.read_csv('/content/verified/rejected.csv')
print(f'Approved: {len(appr)}')
print(f'Rejected: {len(rej)}')
print()
print('=== Approved by pair ===')
print(appr.groupby(['object','color']).size())
print()
print('=== Rejection reasons ===')
print(rej['rejection_reason'].value_counts())

## 7. Save to Drive

In [ ]:
import shutil
DRIVE_DEST = '/content/drive/MyDrive/binding-research/finetuning/verified'
shutil.copytree('/content/verified', DRIVE_DEST, dirs_exist_ok=True)
print(f'Saved to {DRIVE_DEST}')